# 12 - Hyperparameter Tuning

## Objective

Improve Marketing Mix Model performance by tuning regularized regression models.

### Models
- Ridge
- Lasso
- ElasticNet

### Techniques
- GridSearchCV
- RandomizedSearchCV
- Learning Curves
- Validation Curves
- Hyperparameter comparison

> In production, hyperparameter tuning helps improve generalization and prevents overfitting.


In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    RandomizedSearchCV,
    learning_curve,
    validation_curve
)
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.metrics import r2_score, mean_squared_error

ROOT = Path.cwd()
DATA = ROOT/"data"/"processed"/"marketing_mix_model_ready.csv"

df = pd.read_csv(DATA)

X = df.drop(columns=["Sales"])
y = df["Sales"]

X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.2,random_state=42
)


## 1. Ridge Grid Search

In [ ]:

ridge_grid={"alpha":[0.01,0.1,1,10,50,100]}

ridge_search=GridSearchCV(
    Ridge(),
    ridge_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

ridge_search.fit(X_train,y_train)

print("Best Params:",ridge_search.best_params_)
print("Best CV Score:",ridge_search.best_score_)


## 2. Lasso Random Search

In [ ]:

lasso_params={"alpha":np.logspace(0,5,50)}

lasso_search=RandomizedSearchCV(
    Lasso(max_iter=20000),
    lasso_params,
    n_iter=15,
    cv=5,
    random_state=42,
    scoring="r2"
)

lasso_search.fit(X_train,y_train)

print("Best Params:",lasso_search.best_params_)
print("Best Score:",lasso_search.best_score_)


## 3. ElasticNet Grid Search

In [ ]:

elastic_grid={
"alpha":[0.01,0.1,1,10],
"l1_ratio":[0.2,0.5,0.8]
}

elastic_search=GridSearchCV(
    ElasticNet(max_iter=20000),
    elastic_grid,
    cv=5,
    scoring="r2"
)

elastic_search.fit(X_train,y_train)

print(elastic_search.best_params_)


## 4. Compare Tuned Models

In [ ]:

models={
"Ridge":ridge_search.best_estimator_,
"Lasso":lasso_search.best_estimator_,
"ElasticNet":elastic_search.best_estimator_
}

rows=[]

for name,m in models.items():
    pred=m.predict(X_test)
    rows.append({
        "Model":name,
        "R2":r2_score(y_test,pred),
        "RMSE":np.sqrt(mean_squared_error(y_test,pred))
    })

comparison=pd.DataFrame(rows).sort_values("R2",ascending=False)
display(comparison)


## 5. Learning Curve

In [ ]:

best=models[comparison.iloc[0]["Model"]]

sizes,train_score,test_score=learning_curve(
    best,
    X,
    y,
    cv=5,
    train_sizes=np.linspace(.1,1,8),
    scoring="r2"
)

plt.figure(figsize=(8,5))
plt.plot(sizes,train_score.mean(axis=1),label="Train")
plt.plot(sizes,test_score.mean(axis=1),label="Validation")
plt.xlabel("Training Samples")
plt.ylabel("R²")
plt.title("Learning Curve")
plt.legend()
plt.grid(True)
plt.show()


## 6. Validation Curve (Ridge Alpha)

In [ ]:

alphas=np.logspace(-2,2,8)

train,val=validation_curve(
    Ridge(),
    X,
    y,
    param_name="alpha",
    param_range=alphas,
    cv=5,
    scoring="r2"
)

plt.figure(figsize=(8,5))
plt.semilogx(alphas,train.mean(axis=1),label="Train")
plt.semilogx(alphas,val.mean(axis=1),label="Validation")
plt.xlabel("Alpha")
plt.ylabel("R²")
plt.title("Validation Curve")
plt.legend()
plt.grid(True)
plt.show()


## 7. Save Best Model

In [ ]:

winner=comparison.iloc[0]["Model"]
print("Selected Model:",winner)

best=models[winner]

coef=pd.DataFrame({
    "Feature":X.columns,
    "Coefficient":best.coef_
})

OUT=ROOT/"models"
OUT.mkdir(exist_ok=True)

coef.to_csv(OUT/"tuned_model_coefficients.csv",index=False)

print("Saved tuned coefficients.")


# Business Insights

- Hyperparameter tuning improves model robustness.
- Ridge is often preferred when marketing channels are highly correlated.
- Lasso can remove weak channels by shrinking coefficients to zero.
- ElasticNet balances feature selection and coefficient stability.
